In [10]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import FortranFile
import smplotlib
from hmf import MassFunction
from astropy.cosmology import FlatLambdaCDM
import pandas as pd

# Read the Luminosity Bin File

In [11]:
inputs = "/Users/dexbhadra/Documents/delphi/input_z4p5Tree_files/"

In [12]:
path_sfr_lum_data = inputs + "lum_all_cueto_kroupa_all_Z_delphi.bin"
with FortranFile(path_sfr_lum_data, "r") as f:
    n_wavelength = int(f.read_ints(np.int64)[0])   #7
    max_lum_step = int(f.read_ints(np.int64)[0])   #30 steps of 30 Myr
    n_redshifts = int(f.read_ints(np.int64)[0])    #44
    n_Z_imf = int(f.read_ints(np.int64)[0])        #100

    table_lum_conv_Z_imf = np.zeros((n_wavelength, max_lum_step, n_redshifts, n_Z_imf))
    for iwave in range(n_wavelength):
        for istep in range(max_lum_step):
            for ired in range(n_redshifts):
                table_lum_conv_Z_imf[iwave, istep, ired, :] = f.read_reals(np.float64)
print(n_wavelength, max_lum_step, n_redshifts, n_Z_imf)
print(table_lum_conv_Z_imf.shape)

7 30 44 100
(7, 30, 44, 100)


In [13]:
with FortranFile(inputs + "redshift_grid_mh8to14.bin", "r") as f:
    n_redshifts = f.read_ints(np.int32)
    redshifts = f.read_reals(np.float64)

# the 100 metallicities
with FortranFile(inputs + "table_cueto_kroupa_imf_m2.bin", "r") as f:
    f.read_ints(np.int64)                 # 100
    Z_init = f.read_reals(np.float64)[0] # -6.0
    Z_step = f.read_reals(np.float64)[0]   # 0.0434
log_Z = Z_init + np.arange(n_Z_imf) * log_Z_step

print(redshifts[:5], redshifts[-1])   
print(log_Z[:3], log_Z[-1])           

[4.5     4.58308 4.66934 4.75899 4.85224] 40.76271
[-6.         -5.95656566 -5.91313131] -1.6999999999999833


In [15]:
L1500 = table_lum_conv_Z_imf[2, 0]          # channel 2 = 1500 A, age step 0 = 0-30 Myr old; shape (44 redshifts, 100 metallicities)
iZ = 67                   
for iz in range(44):
    print(f"z = {redshifts[iz]:.2f}   Z = {10**log_Z[iZ]:.5f}   L1500 = {L1500[iz, iZ]:.4e}")

z = 4.50   Z = 0.00081   L1500 = 2.4011e+40
z = 4.58   Z = 0.00081   L1500 = 2.4124e+40
z = 4.67   Z = 0.00081   L1500 = 2.4239e+40
z = 4.76   Z = 0.00081   L1500 = 2.4358e+40
z = 4.85   Z = 0.00081   L1500 = 2.4479e+40
z = 4.95   Z = 0.00081   L1500 = 2.4605e+40
z = 5.05   Z = 0.00081   L1500 = 2.4733e+40
z = 5.16   Z = 0.00081   L1500 = 2.4866e+40
z = 5.27   Z = 0.00081   L1500 = 2.5002e+40
z = 5.38   Z = 0.00081   L1500 = 2.5143e+40
z = 5.50   Z = 0.00081   L1500 = 2.5287e+40
z = 5.63   Z = 0.00081   L1500 = 2.5437e+40
z = 5.76   Z = 0.00081   L1500 = 2.5591e+40
z = 5.90   Z = 0.00081   L1500 = 2.5751e+40
z = 6.05   Z = 0.00081   L1500 = 2.5916e+40
z = 6.20   Z = 0.00081   L1500 = 2.6087e+40
z = 6.37   Z = 0.00081   L1500 = 2.6264e+40
z = 6.54   Z = 0.00081   L1500 = 2.6448e+40
z = 6.73   Z = 0.00081   L1500 = 2.6640e+40
z = 6.92   Z = 0.00081   L1500 = 2.6839e+40
z = 7.13   Z = 0.00081   L1500 = 2.7046e+40
z = 7.35   Z = 0.00081   L1500 = 2.7262e+40
z = 7.59   Z = 0.00081   L1500 =

The IMF starts rising or turns top-heavy once the L1500 values change. eSFE and fiducial models need a IMF that is Krouppa and not a function of redshift and metallicity. So we need to find the metallicity at z=4.5 where f_massive <=0 in order to be able to replicate the Krouppa IMF.

In [16]:
def f_massive(log_Z, z):            #
    x = 1 + log_Z - np.log10(0.0134)
    return 1.07 * (1 - 2**x) + 0.04 * 2.67**x * z

kroupa = f_massive(log_Z[None, :], redshifts[:, None]) <= 0     #n_redshifts x n_Z_imf, True = Kroupa
print("Kroupa cells:", kroupa.sum(), "of", kroupa.size)
print("Kroupa metallicities at z = 4.5:", kroupa[0].sum(), "-> Z from", 10**log_Z[kroupa[0]][0], "to", 10**log_Z[kroupa[0]][-1])

Kroupa cells: 488 of 4400
Kroupa metallicities at z = 4.5: 21 -> Z from 0.002699685234971679 to 0.019952623149689562


In [17]:
kroupa_rows = f_massive(log_Z, redshifts[0]) <= 0 # True for the 21 metallicities where the IMF is Kroupa at z = 4.5
Z_starburst = 10**log_Z[kroupa_rows]   #0.0027 ... 0.02 (metallicity values as Z)
n_Z_kroupa = len(Z_starburst)          #21
print(n_Z_kroupa, Z_starburst)

21 [0.00269969 0.00298365 0.00329748 0.00364432 0.00402764 0.00445128
 0.00491948 0.00543693 0.0060088  0.00664083 0.00733933 0.00811131
 0.00896448 0.0099074  0.01094949 0.0121012  0.01337404 0.01478077
 0.01633546 0.01805368 0.01995262]


In [20]:
with FortranFile(inputs + "lum_kroupa_from_cueto_table.bin", "w") as f:
    f.write_record(np.int32(n_wavelength))       # 7 channels                       (line 181)
    f.write_record(np.int32(max_lum_step))       # 30 age steps                     (line 183, must be 30)
    f.write_record(np.int32(n_redshifts))          # 44 redshifts                     (line 189, must match the trees)
    f.write_record(np.int32(n_Z_kroupa))   # 21 metallicities                 (line 215)
    # record 5: the metallicities                                             (line 217)
    f.write_record(Z_starburst)
    #one record of the 30 age steps at z = 4.5   (lines 219-223)
    for iZ in np.where(kroupa_rows)[0]:     # indices of the 21 Kroupa rows
        for iwave in range(n_wavelength):
            f.write_record(table_lum_conv_Z_imf[iwave, :, 0, iZ])